# DDP: синхронизация градиентов

`DistributedDataParallel` хранит полную копию модели на каждом ранке. После локального `backward` градиенты одного параметра могут отличаться, потому что ранки обработали разные части батча. DDP выполняет all-reduce и усредняет их, так что следующий шаг оптимизатора на всех ранках использует одинаковые градиенты.

## Что происходит в настоящем DDP

На практике DDP не ждёт готовности всех градиентов, чтобы отправить их одной большой операцией. Параметры объединяются в **бакеты**: когда backward заполнил очередной бакет, его all-reduce может выполняться параллельно с вычислением более ранних слоёв. Такой overlap скрывает часть коммуникационных затрат.

В этой задаче бакетов и асинхронных операций нет. Мы моделируем только итоговую семантику для каждого параметра:

$$g_k = \frac{1}{N} \sum_{r=0}^{N-1} g_{r,k},$$

где $N$ — число ранков, а $g_{r,k}$ — локальный градиент параметра $k$ на ранке $r$.

## Задача

Реализуйте `ddp_gradient_sync(rank_grads)`. Вход — непустой список словарей, по одному словарю градиентов на ранг. Верните один словарь усреднённых градиентов.

Требования:

- наборы ключей и shape соответствующих тензоров должны совпадать на всех ранках, иначе `ValueError`;
- если градиент параметра равен `None` на всех ранках, верните для него `None`;
- если `None` встретился только на части ранков, выбросьте `ValueError`;
- пустой список ранков некорректен и должен приводить к `ValueError`;
- сохраняйте dtype и shape градиента; для fp16/bf16 допустимо и ожидаемо считать среднее в float32, затем привести результат обратно;
- не используйте `torch.distributed`.

In [ ]:
import torch


def ddp_gradient_sync(
    rank_grads: list[dict[str, torch.Tensor | None]],
) -> dict[str, torch.Tensor | None]:
    # TODO: validate rank dictionaries and average each gradient.
    raise NotImplementedError


<details><summary>Tip 1</summary>

Что именно DDP синхронизирует и в какой момент шага?

</details>

<details><summary>Tip 2</summary>

Почему average, а не sum? Подумай про эквивалентность большому батчу.

</details>

<details><summary>Tip 3</summary>

Схема: stack по ранкам → mean(dim=0). Перед этим проверь одинаковые ключи и shape, а также отдельно обработай None-кейсы.

</details>

In [ ]:
from torch_judge import check

try:
    check('ddp_gradient_sync')
except Exception as error:
    print(f'Judge could not run yet: {error}')
